<a href="https://colab.research.google.com/github/baortraam/Scikit-learn-projects/blob/master/Cyber_Security_Attack_Using_Network_Traffic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cyber Security Attack Using Network Traffic**

**Step 1: Import the dependencies**

In [37]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report

**Step 2: Data preprocessing**

In [2]:
df = pd.read_csv('/content/cyber attack dataset.csv', encoding = 'latin-1')
df

,duration,src_bytes,dst_bytes,packet_count,protocol,failed_logins,attack_type
0,1,8605,418,631,TCP,0,DDoS
1,1,499,148,131,UDP,0,PortScan
2,10,370,160,105,UDP,0,PortScan
3,2,5138,320,666,TCP,0,DDoS
4,36,524,467,58,UDP,10,BruteForce
...,...,...,...,...,...,...,...
99995,10,380,41,143,TCP,0,PortScan
99996,2,153,146,85,TCP,0,PortScan
99997,37,161,114,61,UDP,10,BruteForce
99998,4,142,69,73,TCP,0,PortScan


In [3]:
df.describe()

,duration,src_bytes,dst_bytes,packet_count,failed_logins
count,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000
mean,16.462720,2219.62870,426.171030,202.550770,1.630520
std,15.632207,2751.14241,466.528205,249.729802,3.045577
min,1.000000,50.00000,20.000000,5.000000,0.000000
25%,4.000000,341.00000,131.000000,37.000000,0.000000
50%,10.000000,781.00000,256.000000,87.000000,0.000000
75%,28.000000,3024.00000,458.000000,257.000000,3.000000
max,60.000000,10000.00000,2000.000000,1000.000000,10.000000


In [11]:
cat_cols = ['protocol']

In [12]:
num_cols = ['duration',	'src_bytes', 'dst_bytes',	'packet_count', 'failed_logins']

In [14]:
df['protocol'].unique()

array(['TCP', 'UDP'], dtype=object)

In [17]:
cat_pipeline = Pipeline(steps = [
    ('ordinal-encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [20]:
num_pipeline = Pipeline(steps = [
    ('scaler', StandardScaler())
])

In [22]:
col_transformer = ColumnTransformer(transformers=[
    ('cat_pipeline', cat_pipeline, cat_cols),
    ('num_pipeline', num_pipeline, num_cols)],
    remainder = 'drop',
    n_jobs = -1
)

In [24]:
df['attack_type'].unique()

array(['DDoS', 'PortScan', 'BruteForce', 'Normal'], dtype=object)

In [31]:
le = LabelEncoder()

In [33]:
df['attack_type_encoded'] = le.fit_transform(df['attack_type'])

In [34]:
df

,duration,src_bytes,dst_bytes,packet_count,protocol,failed_logins,attack_type,attack_type_encoded
0,1,8605,418,631,TCP,0,DDoS,1
1,1,499,148,131,UDP,0,PortScan,3
2,10,370,160,105,UDP,0,PortScan,3
3,2,5138,320,666,TCP,0,DDoS,1
4,36,524,467,58,UDP,10,BruteForce,0
...,...,...,...,...,...,...,...,...
99995,10,380,41,143,TCP,0,PortScan,3
99996,2,153,146,85,TCP,0,PortScan,3
99997,37,161,114,61,UDP,10,BruteForce,0
99998,4,142,69,73,TCP,0,PortScan,3


In [67]:
df['attack_type_encoded'].value_counts()

,count
attack_type_encoded,
1,25077
0,25039
3,25008
2,24876


In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   duration             100000 non-null  int64 
 1   src_bytes            100000 non-null  int64 
 2   dst_bytes            100000 non-null  int64 
 3   packet_count         100000 non-null  int64 
 4   protocol             100000 non-null  object
 5   failed_logins        100000 non-null  int64 
 6   attack_type          100000 non-null  object
 7   attack_type_encoded  100000 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 6.1+ MB


In [55]:
X = df.iloc[:, 0:6]
y = df.iloc[:, 7]

In [56]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=11,test_size=0.2, stratify=y)

**Step 3: Training model**

In [27]:
knn = KNeighborsClassifier()

In [71]:
param_grid = {
    'kneighborsclassifier__n_neighbors': [15, 21, 31],
     'kneighborsclassifier__weights': ['uniform'],
     'kneighborsclassifier__metric': ['euclidean', 'manhattan']
}

In [72]:
pipefinal = make_pipeline(col_transformer, knn)

In [73]:
grid_search = GridSearchCV(pipefinal,
                           param_grid,
                           cv=2,
                           scoring = 'accuracy',
                           n_jobs=-1)

In [74]:
grid_search.fit(X_train, y_train)

GridSearchCV(cv=2,
             estimator=Pipeline(steps=[('columntransformer',
                                        ColumnTransformer(n_jobs=-1,
                                                          transformers=[('cat_pipeline',
                                                                         Pipeline(steps=[('ordinal-encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore',
                                                                                                        sparse_output=False))]),
                                                                         ['protocol']),
                                                                        ('num_pipeline',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['duration',
                                                                          'src_bytes',
                                                                          'dst_bytes',
                                                                          'packet_count',
                                                                          'failed_logins'])])),
                                       ('kneighborsclassifier',
                                        KNeighborsClassifier())]),
             n_jobs=-1,
             param_grid={'kneighborsclassifier__metric': ['euclidean',
                                                          'manhattan'],
                         'kneighborsclassifier__n_neighbors': [15, 21, 31],
                         'kneighborsclassifier__weights': ['uniform']},
             scoring='accuracy')

In [75]:
grid_search.best_score_

np.float64(0.9995125)

In [76]:
grid_search.score(X_test, y_test)

0.9995

In [77]:
grid_search.score(X_train, y_train)

0.99965

In [78]:
y_pred = grid_search.predict(X_test)

**Step 4: Model evaluation**

In [79]:
print(confusion_matrix(y_pred, y_test))

[[5008    0    0    0]
 [   0 5015    0    0]
 [   0    0 4965    0]
 [   0    0   10 5002]]


In [80]:
print(classification_report(y_pred, y_test))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5008
           1       1.00      1.00      1.00      5015
           2       1.00      1.00      1.00      4965
           3       1.00      1.00      1.00      5012

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000

